# K-Means Clustering - Allrecipes.com

### Question:
How effectively can K-Means clustering be applied to segment recipes, based on macronutrient composition and total recipe recreation time, to support time-constrained individuals in making nutritionally informed choices?

#### Overview:
To answer the question above, the [all_recipes.csv](https://github.com/owlzyseyes/tastyR/tree/main/data-raw) data, which was originally scraped from [www.allrecipes.com](https://www.allrecipes.com/), and featured on [https://www.brians.works/i-scraped-14k-recipes-so-you-wont-have-to/](https://www.brians.works/i-scraped-14k-recipes-so-you-wont-have-to/), was provided to [Github](https://github.com/) by [owlzeyes - Brian Mubia](https://github.com/owlzyseyes) and for this project was taken from the [tastyR package](https://github.com/owlzyseyes/tastyR) (see also [tastyR package](https://cran.r-project.org/package=tastyR)).

The original [all_recipes.csv](https://raw.githubusercontent.com/owlzyseyes/tastyR/refs/heads/main/data-raw/allrecipes.csv) file contains 14,426 rows and 16 columns of data. This data was chosen, rather than the smaller more curated [cuisines.csv](https://raw.githubusercontent.com/owlzyseyes/tastyR/refs/heads/main/data-raw/cuisines.csv) file due to the requirement for demonstrating data transformation and cleansing skills. The smaller file contains 2,218 rows and 17 columns of data with the additional column being a character column 'country - The country/region the cuisine is from' and won't be used in this analysis.

TODO: *Write an overview/summary of the steps taken in here when it's finished.*

## 1. Libraries and Data Loading

In [1]:
import warnings
warnings.filterwarnings("ignore") # suppress non-critical warnings throughout the script.

# Import today's date to use in the work.
from datetime import date

In [2]:
# Standard library imports
import importlib
import subprocess
import sys

# List required packages
required_packages = [
    "pandas",          # DataFrames, joins, light transforms
    "numpy",           # Numerical ops
    "matplotlib",      # Plotting
    "seaborn",         # Statistical plots
    "sklearn",         # ML
    "statsmodels",     # Statistical modelling
    "ydata_profiling", # EDA (optional)
    "ydata-sdk",       # Advanced data profiling, validation, and synthetic data
    "IPython.display",
]

"""
The ensure_packages function below is set up to check whether each package in the above list is installed.
- If installed it prints a confirmation message with a tick.
- If missing it installs the package using pip install. 
"""
def ensure_packages(packages):
    for pkg in packages:
        try:
            importlib.import_module(pkg)
            print(f"✓ {pkg} already installed")
        except ImportError:
            print(f"✗ {pkg} missing — installing...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

# Run the package check.
ensure_packages(required_packages)

✓ pandas already installed
✓ numpy already installed
✓ matplotlib already installed
✓ seaborn already installed
✓ sklearn already installed
✓ statsmodels already installed


✓ ydata_profiling already installed
✗ ydata-sdk missing — installing...
✓ IPython.display already installed


In [3]:
# Import the libraries required for this work
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from IPython.display import HTML


In [4]:
df = pd.read_csv('https://raw.githubusercontent.com/owlzyseyes/tastyR/refs/heads/main/data-raw/allrecipes.csv')
df.head(10)

,name,url,author,date_published,ingredients,calories,fat,carbs,protein,avg_rating,total_ratings,reviews,prep_time,cook_time,total_time,servings
0,Chewy Whole Wheat Peanut Butter Brownies,https://www.allrecipes.com/recipe/140717/chewy...,DMOMMY,2020-06-18,"⅓ cup margarine, softened, ⅔ cup white sugar, ...",222.0,13.0,24.0,6.0,4.4,47.0,36.0,20,35,55,16.0
1,Pumpkin Pie Eggnog,https://www.allrecipes.com/recipe/269204/pumpk...,Bobbie Susan,2022-09-26,"12 egg yolks, 2 cups heavy whipping cream, ½ ...",477.0,31.0,43.0,8.0,5.0,1.0,1.0,10,5,495,8.0
2,Eggs Poached in Tomato Sauce,https://www.allrecipes.com/recipe/238054/eggs-...,Bren,2018-06-08,"2 tablespoons olive oil, or to taste, ½ onion...",354.0,18.0,32.0,20.0,4.8,4.0,4.0,10,75,85,4.0
3,Minestrone Casserole,https://www.allrecipes.com/minestrone-casserol...,Sarah Brekke,2025-03-03,4 cups dried mafalda pasta (mini lasagna noodl...,356.0,9.0,53.0,19.0,4.3,14.0,13.0,20,40,60,8.0
4,Yummy Stuffed Peppers,https://www.allrecipes.com/recipe/241937/yummy...,Procrastigirl,2024-12-11,"4 green bell peppers, halved lengthwise and se...",366.0,22.0,23.0,19.0,4.7,84.0,67.0,30,95,125,8.0
5,Prime Rib Our Way,https://www.allrecipes.com/recipe/219586/prime...,Cajun Girl,2019-04-03,"1 (8 pound) standing rib roast, fat trimmed, 2...",709.0,47.0,31.0,37.0,4.2,5.0,3.0,45,80,155,12.0
6,Parmesan Chicken Legs,https://www.allrecipes.com/recipe/8636/parmesa...,Anika,2023-01-04,"1 large egg, 2 cups grated Parmesan cheese, 1 ...",466.0,27.0,1.0,52.0,4.4,648.0,468.0,15,45,60,6.0
7,Chicken Andouille Gumbo,https://www.allrecipes.com/recipe/12950/chicke...,Bob Cody,2022-07-14,"12 cups water, 3 pounds chicken parts, 2 table...",782.0,61.0,19.0,40.0,4.6,347.0,259.0,10,190,200,8.0
8,Sweet Pork for Burritos,https://www.allrecipes.com/recipe/185816/sweet...,Dean,2023-01-19,"3 pounds pork shoulder roast, 2 cups salsa, 1 ...",355.0,15.0,33.0,23.0,4.7,129.0,102.0,30,480,510,12.0
9,Quick Baked Chicken Parmesan,https://www.allrecipes.com/recipe/239507/quick...,ChristineM,2024-11-14,"canola oil cooking spray, ½ cup water, 1 egg,...",395.0,12.0,33.0,37.0,4.7,195.0,153.0,20,55,75,6.0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14426 entries, 0 to 14425
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   name            14426 non-null  object 
 1   url             14426 non-null  object 
 2   author          14426 non-null  object 
 3   date_published  14426 non-null  object 
 4   ingredients     14417 non-null  object 
 5   calories        14226 non-null  float64
 6   fat             14070 non-null  float64
 7   carbs           14212 non-null  float64
 8   protein         14178 non-null  float64
 9   avg_rating      13454 non-null  float64
 10  total_ratings   13454 non-null  float64
 11  reviews         13353 non-null  float64
 12  prep_time       14426 non-null  int64  
 13  cook_time       14426 non-null  int64  
 14  total_time      14426 non-null  int64  
 15  servings        14405 non-null  float64
dtypes: float64(8), int64(3), object(5)
memory usage: 1.8+ MB


## 2. EDA - Basic

#### Duplication
Check for full duplication before conducting any other analysis of the data.

In [6]:
df[df.duplicated(keep=False)]

,name,url,author,date_published,ingredients,calories,fat,carbs,protein,avg_rating,total_ratings,reviews,prep_time,cook_time,total_time,servings


There are no complate duplicates in the data however some recipes share the same name as others (see below). From observing the other columns in the dataset, especially the ingredients list, it can be seen that these are a duplicate dish name but not a duplicate recipe and therefore none will be removed.

In [7]:
dupe_name = df[df['name'].duplicated(keep=False)]
dupe_name.sort_values(by='name', ascending=False).head(8)


,name,url,author,date_published,ingredients,calories,fat,carbs,protein,avg_rating,total_ratings,reviews,prep_time,cook_time,total_time,servings
1827,Tito's Bloody Mary,https://www.allrecipes.com/recipe/284513/titos...,Tito's Handmade Vodka,2024-11-30,"1 ½ ounces Tito's Handmade Vodka, 4 ounces of ...",120.0,0.0,6.0,1.0,NaN,NaN,NaN,5,0,5,1.0
11065,Tito's Bloody Mary,https://www.allrecipes.com/recipe/8393460/tito...,Tito's Handmade Vodka,2022-05-18,"1 ½ ounces Tito's Handmade Vodka, 4 ounces you...",122.0,NaN,6.0,1.0,1.0,1.0,1.0,5,0,5,1.0
7650,Strawberry Rhubarb Cobbler,https://www.allrecipes.com/recipe/233121/straw...,cstillings24,2025-07-24,"cooking spray, 6 cups sliced rhubarb, ⅔ cup wh...",239.0,9.0,38.0,2.0,5.0,3.0,3.0,10,50,75,12.0
5570,Strawberry Rhubarb Cobbler,https://www.allrecipes.com/recipe/233075/straw...,bheavnly,2025-04-30,"8 cups chopped rhubarb, 1 cup sliced fresh str...",305.0,8.0,57.0,3.0,4.7,68.0,60.0,10,45,55,12.0
5401,Sriracha Deviled Eggs,https://www.allrecipes.com/sriracha-deviled-eg...,Yolanda Gutierrez,2024-10-22,"3 large eggs, hard boiled, peeled, 2 tablespoo...",143.0,12.0,1.0,6.0,4.0,1.0,NaN,10,15,25,3.0
12448,Sriracha Deviled Eggs,https://www.allrecipes.com/recipe/236516/srira...,Jenny Saunders,2025-03-24,"12 large eggs, 2 tablespoons mayonnaise, or as...",90.0,7.0,1.0,6.0,4.6,29.0,21.0,20,15,50,12.0
7077,Seafood Paella,https://www.allrecipes.com/seafood-paella-reci...,Roscoe Hall,2024-10-23,"1/4 cup olive oil, 1 yellow onion, diced, 2 re...",665.0,25.0,36.0,65.0,5.0,3.0,3.0,20,30,55,6.0
14265,Seafood Paella,https://www.allrecipes.com/recipe/12840/paella...,Dell,2024-05-02,"20 mussels, cleaned and debearded, 10 large s...",367.0,6.0,52.0,23.0,4.6,5.0,4.0,20,25,45,10.0


In [8]:
# Count the number of duplicates in each column.
dupe_counts = df.apply(lambda col: col.duplicated().sum())
dupe_counts

name                 25
url                   0
author             6129
date_published    12884
ingredients          14
calories          13313
fat               14299
carbs             14245
protein           14311
avg_rating        14395
total_ratings     13576
reviews           13608
prep_time         14380
cook_time         14238
total_time        13976
servings          14346
dtype: int64

After calculating the number of values that appear more than once we can be satisfied that any duplication present is acceptable. URLs are not duplicated, very few of the ingredients lists are duplicated but as they are for different URLs they are for different recipes. All other duplicated columns are acceptable due to the nature of the fields (date, calories, fat, carbs, protein, avg_rating, total_ratings, number of reviews, prep time, cook time, total time and number of servings).

#### Unique Identifier and Column Types

K-Means clustering is a distance based model therefore only works with numeric, scaled data. The data will be split into df_kmeans and df_features after EDA and data cleansing. After K-Means clustering we will generate some labels/descriptions for the recipes from the features columns, therefore a unique identifier will be required to act as a joining key between the two datasets. The original data has no unique_id, therefore we create one now. We also convert each url into a clickable link to allow for easier validation of the data against the source data within the Allrecipes website. Here we note that some information may have changed after the data was extracted by the creator of the original dataset used in this analysis.

In [9]:
# Create a unique_id for each row.
df = df.reset_index(drop=True)
df['unique_id'] = df.index

# Move 'unique_id' to the first position
cols = ['unique_id'] + [col for col in df.columns if col != 'unique_id']
df = df[cols]

# Wrap each url in an <a> tag, and use the data from "name" text displayed in the link.
df['url'] = df.apply(
    lambda row: f'<a href="{row["url"]}" target="_blank">{row["name"]}</a>',
    axis=1
)

# Can use the following code line to drop the name column, but for now leave it in so that when the HTML rendering is not used we can still see the name
# alongside the url.
#df = df.drop('name', axis=1)
# Then tell pandas to render HTML - Note this format should be used to when the clickable links are needed rather than the plain DataFrame.
HTML(df.head(5).to_html(escape=False))

,unique_id,name,url,author,date_published,ingredients,calories,fat,carbs,protein,avg_rating,total_ratings,reviews,prep_time,cook_time,total_time,servings
0,0,Chewy Whole Wheat Peanut Butter Brownies,Chewy Whole Wheat Peanut Butter Brownies,DMOMMY,2020-06-18,"⅓ cup margarine, softened, ⅔ cup white sugar, ½ cup packed brown sugar, 2 eggs, 1 cup peanut butter, ½ teaspoon vanilla extract, 2 tablespoons water, ¾ cup whole wheat flour, ¼ cup all-purpose flour, ¼ teaspoon salt, 1 teaspoon baking powder, ¼ teaspoon baking soda",222.0,13.0,24.0,6.0,4.4,47.0,36.0,20,35,55,16.0
1,1,Pumpkin Pie Eggnog,Pumpkin Pie Eggnog,Bobbie Susan,2022-09-26,"12 egg yolks, 2 cups heavy whipping cream, ½ teaspoon vanilla extract, 1 (15 ounce) can pumpkin puree, ½ cup light brown sugar, ½ cup white sugar, ¼ cup maple syrup, ¼ cup evaporated milk, 1 teaspoon pumpkin pie spice, ¼ teaspoon ground cinnamon, 1 pinch salt, 2 cups milk, 1 tablespoon brandy, or as needed",477.0,31.0,43.0,8.0,5.0,1.0,1.0,10,5,495,8.0
2,2,Eggs Poached in Tomato Sauce,Eggs Poached in Tomato Sauce,Bren,2018-06-08,"2 tablespoons olive oil, or to taste, ½ onion, finely chopped, 2 cloves garlic, finely chopped, 8 cups tomato sauce, ¼ cup dry red wine, or more to taste, 1 tablespoon dried parsley, 1 tablespoon dried basil, 1 tablespoon dried oregano, ½ teaspoon salt, ¼ teaspoon freshly ground black pepper, or more to taste, 1 bay leaf, or more to taste, 1 pinch red pepper flakes, 8 eggs",354.0,18.0,32.0,20.0,4.8,4.0,4.0,10,75,85,4.0
3,3,Minestrone Casserole,Minestrone Casserole,Sarah Brekke,2025-03-03,"4 cups dried mafalda pasta (mini lasagna noodles), 2 tablespoons olive oil, 2 carrots, sliced, 2 stalks celery, sliced, 1 onion, chopped, 1 zucchini, quartered lengthwise and sliced, 2 cloves garlic, minced, 1 (15-ounce) can red kidney beans, 1 (15-ounce) can cannellini beans, 1 (15-ounce) can diced tomatoes with Italian herbs, 1 (15-ounce) can tomato sauce, 1 cup frozen cut green beans, 1/2 cup grated Parmesan cheese, divided, 1 teaspoon Italian seasoning, 1/2 teaspoon salt, 1/4 teaspoon freshly ground black pepper, 1 cup shredded mozzarella cheese, chopped fresh basil and/or oregano",356.0,9.0,53.0,19.0,4.3,14.0,13.0,20,40,60,8.0
4,4,Yummy Stuffed Peppers,Yummy Stuffed Peppers,Procrastigirl,2024-12-11,"4 green bell peppers, halved lengthwise and seeded, 2 tablespoons olive oil, divided, 1 ¼ cups water, divided, ½ cup rice, 1 cube beef bouillon, 1 white onion, diced, 2 cloves garlic, minced, 1 pound ground beef, ¼ teaspoon onion powder, ¼ teaspoon garlic powder, ¼ teaspoon dried oregano, 1 (14.5 ounce) can diced tomatoes, 1 (8 ounce) package shredded Cheddar cheese, divided, ¼ cup grated Parmesan cheese, ½ cup ketchup, ½ cup Worcestershire sauce",366.0,22.0,23.0,19.0,4.7,84.0,67.0,30,95,125,8.0


#### Numeric column EDA
Use df.describe() to display summary statistics on all numeric columns within the data.

In [10]:
df.describe()

,unique_id,calories,fat,carbs,protein,avg_rating,total_ratings,reviews,prep_time,cook_time,total_time,servings
count,14426.000000,14226.000000,14070.000000,14212.000000,14178.000000,13454.000000,13454.000000,13353.000000,14426.000000,14426.000000,14426.000000,14405.000000
mean,7212.500000,344.878532,17.835537,32.863425,14.422344,4.525606,102.620262,94.418483,17.345002,42.515943,144.065645,11.028393
std,4164.571827,250.020621,16.679835,27.590609,17.528592,0.409127,172.979317,164.522217,24.872135,96.819185,874.250121,13.026444
min,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000
25%,3606.250000,181.000000,7.000000,14.000000,3.000000,4.400000,5.000000,5.000000,10.000000,10.000000,30.000000,4.000000
50%,7212.500000,307.000000,14.000000,29.000000,8.000000,4.600000,26.000000,24.000000,15.000000,20.000000,55.000000,8.000000
75%,10818.750000,453.000000,24.000000,45.000000,22.000000,4.800000,112.000000,100.000000,20.000000,45.000000,100.000000,12.000000
max,14425.000000,9538.000000,612.000000,746.000000,939.000000,5.000000,997.000000,999.000000,2160.000000,4325.000000,60485.000000,300.000000


There are **14426** recipes in total. Some of these numeric columns are not based on nutrition or time such as avg_rating, total_ratings and reviews. These should be removed and placed into the df_features table rather than form part of the analysis here.

#### Time Based EDA

Time is a key component of our K-Means clustering model, and is required for answering the question which specifically mentions *total recipe recreation time* and *time-constrained individuals*. There are 3 time based columns in the data, here we investigate which are needed (prep time, cook time or total time) and conduct some feature engineering where necessary.

Where prep time and cook time did not sum to total time, revealed that there is a missing time based field from our data that is available on the website, as the website shows additional time needed when following the recipe. Additional time was observed to represent things such as periods required in the meal preparation for activities such as marinating, or defrosting and post-cooking for activities such as resting and cooling. For examples see [Pumpkin Pie Eggnog](https://www.allrecipes.com/recipe/269204/pumpkin-pie-eggnog/) and [Prime Rib Our Way](https://www.allrecipes.com/recipe/219586/prime-rib-our-way/). The decision here is to calculate the additional time by subtracting the prep time and cook time from total time, because all 3 times may affect whether or not an individual chooses a recipe.

Prep time, cook time and additional time are independant stages of the recipe process, but as total time is determined by all of those stages and extra time is calculated from total time, prep time and cook time, the variables are dependant upon each other, and multicollinearity within our data will exist and distort the distance-based clustering as time would be double-counted. Total time becomes a redundant fiels once we have additional time calculated and therefore it is will not be taken into the K-Means dataset and the other 3 time based fields will be retained for K-Means.

Bringing 3 time based fields into the K-Means clustering has the effect of giving time 3 times the influence in the distance calculations compared to the other variables. The three remaining times are all meaningful to the recipe recreation process and bear a real world influence on people's lifestyles, so we reduce the weighting for time by engineering two new time based features.  When considering the participation of the cook/chef in this process they are completing:
1. active tasks, where effort is required for preparation and cooking **{active_time = prep_time + cook_time}**
2. passive tasks, revolved around waiting and lifestyle flexibility **{passive_time = total_time - (prep_time + cook_time)}**  Note: We briefly considered calling this one extra_time

The approach of calculating time based behaviours, means that the original 3 time columns can be dropped, we will reduce the time dominance within the clusters from 3 fields to 2 fields, and it will also enable us to derive rich behavioural insight into the model. We will be able to use these variables to distinguish between different behaviours, for example:
- High-Active/Low-Passive = Labour intensive cooking.
- Low-Active/Low-Passive = Quick meals.
- High-Active/High-Passive = Complex and slow recipes.
- Low-Active/High-Passive = Slow but easy meals.

Next we create the new time-based features.

In [12]:
# Create active_time
df['active_time'] = df['prep_time'] + df['cook_time']

# Create passive_time
df['passive_time'] = df["total_time"] - df['prep_time'] - df['cook_time']

# TODO: For some reason some of the times are in the thousands and so passive time becomes a large negative number. 

[Pumpkin cake](https://www.allrecipes.com/recipe/26061/pumpkin-spice-cake-i/)
the website shows 80 mins in prep+cook+additional. Total = 1hr10 so they are not equal, but my df shows 3020 active and -2940 passive.

This needs thought and sorting out by removing recipes early on where the times were large.

In [ ]:

HTML(df.sort_values(by='passive_time', ascending=True).to_html(escape=False))

Some recipes show zero time in all time based fields, for example [Caramel Corn I](https://www.allrecipes.com/recipe/9386/caramel-corn-i/). Passive time is generated from total time therefore it will never be greater than zero when total time equals zero. Some recipes may be for foods where prep time is greater than zero but cook time equals zero because there is no cooking involved, such as for foods like [Berry Overnight Oats](https://www.allrecipes.com/recipe/257039/berry-overnight-oats/), some recipes may show prep time equal to zero but cook time greater than zero such as [Fruity Tutti Turkey Brine](https://www.allrecipes.com/recipe/219474/fruity-tutti-turkey-brine/) it is assumed here that recipe creators have not factored sourcing ingredients into preparation time and any chopping time is included either of the cooking time and additional time. For all of these reasons, recipes are removed when total time equals zero, and retained where prep time and/or cook time are greater than zero.

The table below displays the number of recipes where the total time equals zero and will be removed from the process here, **360** recipes.

In [ ]:
df_zero_total = df[df['total_time'] == 0]
df_zero_total.describe()

In [ ]:
# Filter to retain recipes where total_time > 0
df = df[df['total_time'] > 0]

We can now drop the original 3 time based fields as we have our 2 time based behavioural indicators (active_time and passive_time)

In [ ]:
# Drop total_time column
df = df.drop(columns=['total_time', "cook_time", "prep_time"])
# Then tell pandas to render HTML to display the clickable links.
#HTML(df.head(5).to_html(escape=False))

df.describe()

#### Calories and Serving Size

It was initially thought that servings could play a role in the nutrition element, as the calories, fat, carbs and protein values could all be affected by the serving size of the recipe, therefore one strategy could be to calculate the nutritional values per serving. However, upon investigation this did not reign true when checking online sources for similar items, for example compare [Chewy Whole Wheat Peanut Butter Brownies](https://www.allrecipes.com/recipe/140717/chewy-whole-wheat-peanut-butter-brownies/) with **222** calories against [Nutracheck - Calories In Brownies](https://www.nutracheck.co.uk/CaloriesIn/Product/Search?desc=brownie) showing around **250** calories for similar looking sized items.

When investigating the recipes with calorie count over **3000** calories (**7 items**) the calories appear to be for the whole recipe rather than per serving, see [Boeuf En Croute](https://www.allrecipes.com/recipe/213512/boeuf-en-croute/) recipe showing **3274** calories when compared to [Nutracheck - Calories In Beef En Croute](https://www.nutracheck.co.uk/CaloriesIn/Product/Search?desc=beef+en+croute) showing around **550** calories per serving. The disparity in the methods of calculating calories per serving could be due to the nature of the webiste, in that recipes are submitted by site members. The website states that duplication of recipes is handled, ingredients lists are evaluated for completeness and accuracy, serving sizes and yields also checked for accuracy, recipes are reviewed to ensure the recipe is able to be replicated by following the instructions and health claims such as *low-fat/low-carb or paleo etc.* are evaluated, see [Approval and Testing](https://www.allrecipes.com/about-us-6648102#toc-recipe-approval-and-testing). There is no mention of verifying the nutritional information or serving sizes, therefore with the absence of a method available in the site information we proceed to remove recipes using alternative justification.

In [ ]:
# Identifying high calorie meals for finding an upper bound.
df_high_cals = df[df['calories'] > 600]
df_high_cals
# Drop columns to shorten the table
df_high_cals = df_high_cals.drop('ingredients', axis=1)
# Then tell pandas to render HTML to display the clickable links.
HTML(df_high_cals.head(10).to_html(escape=False))

In [ ]:
df_high_cals.describe()

To remove some of the bias in the high calorie items, and attempt to rebalance the calories within the population, an upper bound of **600** calories was chosen for the recipes retained, this was done by comparing recipes at various thresholds until the recipes in the population appeared to balance with those from other online sites. Reducing the maximum calories to **600cal** reduced the population by **1560** recipes. 

The servings column will not be used to in the K-Means clustering and so will be placed in the features dataset.

In [ ]:
# Removing high calorie meals.
df = df[df['calories'] <= 600]
df.head(5)

In [ ]:
df.describe()

#### TODO - Investigate Fat Carbs Protein - Missing values for all numeric cols.
Distribution and outliers in numeric columns.

Missing values for character columns.

Remove anything required - Leave the ratings columns as we aren't sure they will be used in the categorising and other elements.

Then separate the data into 3 dataframes.
- **df_full** (containing all columns apart from total_time as we have the other 3 time columns).
- **df_kmeans** (the columns required for the model building + the unique_id).
- **df_features** (the remaining columns + unique_id).

#### TODO - see above

In [ ]:
df_full = df.copy()
df_full.head(5)

In [ ]:
kmeans_cols_to_keep = [
    'unique_id', 'calories', 'fat', 'carbs', 'protein',
    'prep_time', 'cook_time', 'extra_time'
]

df_kmeans = df_full[kmeans_cols_to_keep]

# Everything except the core columns
cols_features = [c for c in df_full.columns if c not in kmeans_cols_to_keep]
df_features = df_full[['unique_id'] + cols_features]

In [ ]:
df_kmeans.head(5)

In [ ]:
df_features.head(5)

## EDA - df_kmeans TODO - Move this up into the general EDA and do it before splitting data into two, we need to remove recipes where any of our kmeans columns are missing, and will need to remove from both tables, so we should do it earlier.

Look for missing values across the nutrtion and time variables, try to fill missing values in where possible. Check for outliers.

In [ ]:
# Identify columns with null values
columns_with_nulls = df_kmeans.columns[df_kmeans.isnull().any()].tolist()

# Print column names containing null values
print("Columns with null values:", columns_with_nulls)

Three columns contain NaN values, listed above, let's look at them.

In [ ]:
df_kmeans.isnull().mean()

Check how many of the rows with null values are null for all of them.

In [ ]:
filtered_df = df_kmeans[df_kmeans[columns_with_nulls].isna().all(axis=1)]
filtered_df

These would be of no use to our model as they would contain too many missing values to be believable. Recipes can have no fat, no carbs, no protein but not none of all 3. If this contained any recipes we would removing them from our population. However they all have a value in at least one of the columns so we will display them here to assess how to handle them.

In [ ]:
# Code left in, just in case it is needed in future. PRecautionary futureproofing method.
# Example: remove rows where fat, protein, and carbs are all NaN
df_kmeans = df_kmeans[~df_kmeans[columns_with_nulls].isna().all(axis=1)]
df_kmeans


## TODO - EDA on df_features columns, some are character and some are numeric.


In [ ]:
df_features.describe(include='object')

In [ ]:
df_features.describe(include=['float64', 'int64'])

Next Steps:
1. Explore the data, deciding which columns are required.
2. Cleanse the required columns dealing with (missing values etc etc)
3. Move to k-means clustering.
4. Write report.